# 07 · Proyecto final: análisis de un dataset con Apache Spark

**Estudiante:** ...
**Curso:** Big Data
**Fecha:** ...

Este cuaderno es la plantilla del informe final. Cada sección indica qué debe contener; el texto en cursiva se reemplaza por el contenido propio. Las celdas de código incluyen un punto de partida que debe completarse. El informe se redacta en tercera persona y las fuentes se citan con normas APA (American Psychological Association, 2020).

### Criterios generales

- El cuaderno debe ejecutarse completo, sin errores, con *Kernel > Restart Kernel and Run All Cells*.
- Cada resultado debe ir acompañado de una interpretación en una celda Markdown.
- El procesamiento se realiza con Spark; pandas solo se usa para graficar resultados agregados.

## 1. Descripción del dataset

*Describir el origen del dataset (entidad, portal, año), qué representa cada registro, el periodo que cubre y las principales variables. Incluir la referencia en formato APA, por ejemplo:*

> Ministerio de Educación Nacional. (2025). *Estadísticas de educación superior* (Versión 2) [Conjunto de datos]. Portal de Datos Abiertos de Colombia. https://www.datos.gov.co/...

## 2. Preguntas de análisis

*Formular entre tres y cinco preguntas que el análisis busca responder. Por ejemplo: ¿qué categoría concentra la mayor parte de los ingresos? ¿existe estacionalidad? ¿qué variables explican mejor el valor de la variable objetivo?*

1. ...
2. ...
3. ...

## Preparación del entorno

La celda siguiente realiza cuatro tareas:

1. **Ubica la carpeta del taller** con `ubicar_taller()`, que busca `scripts/taller_spark.py` a partir de la carpeta actual. Esto evita errores de ruta del tipo `Input path does not exist: file:/data/...`, que aparecen cuando el cuaderno se abre desde una ubicación distinta, como ocurre en Google Colab, donde la carpeta de trabajo es `/content`. Después fija la carpeta `notebooks/` como carpeta de trabajo.
2. **Solo en Google Colab**: busca el taller en cualquier carpeta de `/content` y en Google Drive; si no lo encuentra, lo descarga con `git clone` desde la dirección indicada en `REPO_URL` (por eso, en Colab, debe reemplazarse `<usuario>` por el usuario de GitHub donde está publicado el taller). También instala las dependencias que falten en Colab.
3. Agrega la carpeta `scripts/` a la ruta de búsqueda de Python, para poder importar el módulo `taller_spark.py`, que contiene las funciones de apoyo del taller.
4. Importa `functions` de PySpark con el alias `F`, convención habitual en la documentación oficial (The Apache Software Foundation [ASF], 2026b).
Finalmente, crea la `SparkSession` mediante `iniciar_spark()`. Esta función verifica que Java sea la versión 17 o 21, utiliza el mismo intérprete de Python en el *driver* y en los *executors* y se conecta al clúster de Docker cuando existe la variable `SPARK_MASTER`; en cualquier otro caso trabaja en modo `local[*]`.

In [ ]:
# Preparación del entorno: se ejecuta siempre en primer lugar
import glob, os, subprocess, sys
from pathlib import Path

# >>> CELDA EDITABLE (solo Google Colab): dirección del repositorio del taller en GitHub
REPO_URL = "https://github.com/<usuario>/taller-apache-spark.git"

def en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def ubicar_taller():
    """Devuelve la carpeta raíz del taller: la que contiene scripts/taller_spark.py."""
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]
    for patron in ("/content/*/scripts/taller_spark.py",                  # clonado en Colab
                   "/content/drive/MyDrive/*/scripts/taller_spark.py",    # copiado en Google Drive
                   "/content/drive/MyDrive/*/*/scripts/taller_spark.py"):
        candidatos += [Path(p).parent.parent for p in sorted(glob.glob(patron))]
    for base in candidatos:
        if (base / "scripts" / "taller_spark.py").exists():
            return base
    if en_colab():
        if "<usuario>" in REPO_URL:
            raise RuntimeError("En Google Colab debe escribirse en REPO_URL la dirección del repositorio "
                               "del taller (reemplazar <usuario>) y volver a ejecutar esta celda.")
        destino = Path("/content") / Path(REPO_URL.rstrip("/")).stem        # nombre del repositorio
        subprocess.run(["git", "clone", "-q", REPO_URL, str(destino)], check=True)
        return destino
    raise FileNotFoundError("No se encontró la carpeta del taller (scripts/taller_spark.py). "
                            "Abrir el cuaderno desde la carpeta notebooks/ del repositorio.")

RAIZ_TALLER = ubicar_taller()
os.chdir(RAIZ_TALLER / "notebooks")              # rutas iguales en cualquier entorno (local, Docker, Colab)
sys.path.insert(0, str(RAIZ_TALLER / "scripts"))

if en_colab():                                    # en Colab se instalan las dependencias que falten
    import importlib.util
    faltan = [m for m in ("openpyxl",) if importlib.util.find_spec(m) is None]
    try:
        import pyspark
        if int(pyspark.__version__.split(".")[0]) < 4:
            faltan.append("pyspark==4.3.0")
    except ImportError:
        faltan.append("pyspark==4.3.0")
    if faltan:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Carpeta del taller:", RAIZ_TALLER)

from taller_spark import *                       # iniciar_spark, cargar_desde_config, perfil, ...
from pyspark.sql import functions as F           # funciones de columnas de Spark

spark = iniciar_spark("07_Proyecto")

## 3. Carga y configuración

*Indicar las opciones de lectura utilizadas (separador, codificación, formato de fecha) y justificar los ajustes realizados en el cuaderno 01.*

In [ ]:
df, cfg = cargar_desde_config(spark, version="original")
roles = clasificar_columnas(df, excluir=cfg["columnas_excluir"], max_categorias=cfg["max_categorias"])
NUM, CAT, FEC = roles["numericas"], roles["categoricas"], roles["fechas"]
resumen_roles(roles)
print(f"Filas: {df.count():,} | Columnas: {len(df.columns)}")

## 4. Calidad de los datos

*Presentar el perfil de calidad, describir los problemas encontrados y las decisiones de limpieza tomadas (cuaderno 04), con el número de registros eliminados en cada paso.*

In [ ]:
perfil(df, roles)

In [ ]:
# Carga de la versión limpia generada en el cuaderno 04
limpio, _ = cargar_desde_config(spark, version="limpia")
print(f"Registros después de la limpieza: {limpio.count():,}")
limpio.createOrReplaceTempView(cfg["nombre_tabla"])

## 5. Análisis exploratorio

*Incluir al menos dos gráficos (histograma, serie de tiempo, barras por categoría) con su interpretación.*

In [ ]:
# Gráfico 1: completar

*Interpretación del gráfico 1.*

## 6. Respuesta a las preguntas con Spark SQL

*Una consulta por pregunta, cada una con su interpretación. Se valoran las funciones de ventana y las subconsultas.*

In [ ]:
# Pregunta 1
spark.sql(f"""
SELECT *
FROM {cfg["nombre_tabla"]}
LIMIT 5
""").show(truncate=False)

*Respuesta a la pregunta 1.*

## 7. Modelo (opcional)

*Si el dataset lo permite, presentar un modelo de regresión, clasificación o agrupamiento: variables utilizadas, métricas frente a la línea base e interpretación. Discutir el riesgo de fuga de datos.*

In [ ]:
# Modelo: completar a partir del cuaderno 06

## 8. Evidencia de ejecución en Spark

*Incluir capturas de la Spark UI (pestañas Jobs, Stages y, si se usó cache, Storage) guardadas en `ejercicios/evidencias/`, y explicar qué operaciones generaron shuffle. Una imagen se inserta en Markdown con:* `![Spark UI](../ejercicios/evidencias/jobs.png)`

## 9. Conclusiones

*Responder de forma sintética las preguntas de la sección 2, indicar las limitaciones del análisis (calidad de los datos, tamaño, sesgos) y proponer trabajo futuro.*

## 10. Referencias

*Listar en orden alfabético todas las fuentes citadas, incluida la del dataset, con formato APA 7.ª edición.*

In [ ]:
spark.stop()

## Referencias

American Psychological Association. (2020). *Publication manual of the American Psychological Association* (7.ª ed.). https://doi.org/10.1037/0000165-000

Ministerio de Tecnologías de la Información y las Comunicaciones. (s.f.). *Portal de datos abiertos de Colombia* [Sitio web]. Recuperado el 9 de noviembre de 2026, de https://www.datos.gov.co/